# 통신망 방호 준비 실험 통합 재현
작성: Codex main agent. 실제 조원의 개인 기여 완료를 뜻하지 않는 AI 연구 보조 자료입니다.

개발 20사례와 소스 고정 이후 확인 32사례를 구분합니다. 같은 그래프의 예산 20%·40%는 하나의 군집으로 처리합니다. 아래 코드는 저장 결과만 분석하며 웹 제출을 새로 만들지 않습니다.

원시 웹 CSV는 agent-1~3/ 및 confirmation/agent-1~3/, 소스 고정 근거는 source-freeze.json, 공통 평가기 대조 결과는 verified_rows.csv와 verification.json입니다. 추가 원시 점수 대조는 저장소 루트에서 `.venv/bin/python results/agent-study/2026-11-22/verify_results.py`로 명시적으로 실행합니다. 정확 탐색 기록은 agent-4/records.json입니다.

담당: main agent — 조건·출처 확인 및 통계 재현

In [1]:
from pathlib import Path
import contextlib, io, json, runpy
HERE = Path.cwd()
if not (HERE / 'verified_rows.csv').exists():
    HERE = HERE / 'results/agent-study/2026-11-22'
functions = runpy.run_path(str(HERE / 'analyze_study.py'))
with contextlib.redirect_stdout(io.StringIO()):
    functions['main']()
report = json.loads((HERE / 'study_summary.json').read_text())
verification = json.loads((HERE / 'verification.json').read_text())
assert verification['rows'] == 312 and verification['sources_verified'] == 6
print('312개 웹 실행 결과, 6개 고정 소스, 52개 사례. 공식 점수 재계산 대조 완료.')

312개 웹 실행 결과, 6개 고정 소스, 52개 사례. 공식 점수 재계산 대조 완료.


담당: main agent — 확인 비교 표. 최적 점수 일치는 연결 쌍·전달 점수 일치이며 비용·간선번호까지 같은 정규 최적해와 구분합니다.

In [2]:
from IPython.display import display, HTML
from html import escape
def show(headers, rows):
    cells=lambda values, tag: '<tr>'+''.join(f'<{tag}>{escape(str(v))}</{tag}>' for v in values)+'</tr>'
    display(HTML('<table>'+cells(headers,'th')+''.join(cells(r,'td') for r in rows)+'</table>'))
confirmation=report['stages']['confirmation']
show(['방법','성공/전체','최적 점수 일치','평균 연결률','시간 중앙값(초)'],
     [[m,f"{s['completed']}/{s['runs']}",s['optimal_score_matches'],f"{s['mean_connected_ratio']:.2%}",f"{s['median_seconds']:.4f}"] for m,s in confirmation['methods'].items()])
show(['대조','승/동/패','평균 연결률 차(%p)','95% 그래프 군집 구간'],
     [[m,f"{s['wins']}/{s['ties']}/{s['losses']}",f"{s['mean_connected_pp_difference']:+.3f}",str([round(v,3) for v in s['connected_pp_95pct_cluster_interval']])] for m,s in confirmation['contrasts'].items()])

방법,성공/전체,최적 점수 일치,평균 연결률,시간 중앙값(초)
betweenness,32/32,5,72.28%,0.0895
connectivity_greedy,32/32,4,83.60%,0.0808
degree,32/32,4,72.19%,0.0855
demand_bundle,32/32,9,85.89%,0.1407
demand_single,32/32,7,85.89%,0.1280
local_search,32/32,4,84.08%,0.0823


대조,승/동/패,평균 연결률 차(%p),95% 그래프 군집 구간
betweenness vs degree,9/18/5,+0.092,"[-5.064, 5.179]"
demand_bundle vs degree,26/6/0,+13.702,"[9.836, 17.681]"
local_search vs connectivity_greedy,4/26/2,+0.473,"[0.0, 1.42]"
demand_bundle vs demand_single,4/26/2,+0.000,"[0.0, 0.0]"


담당: agent 4의 실제 원시 실행을 main agent가 통합 — 정확 탐색의 시간 경계는 웹 제출과 다릅니다.

In [3]:
exact=json.loads((HERE/'agent-4/summary.json').read_text())
show(['방법','완료/8','평가 조합 수','상한 계산 수','총 탐색시간(초)'],
     [[m,s['completed'],s['evaluated_combinations'],s['upper_bound_calls'],round(s['total_search_seconds'],4)] for m,s in exact['methods'].items()])
assert all(s['reference_score_matches']==8 and s['full_tiebreak_matches']==8 for s in exact['methods'].values())
print('가지치기 평가 횟수 감소:',f"{exact['aggregate_evaluation_reduction']:.2%}")
print('각 조건 1회 측정이며 4개 그래프다. 성능 일반화나 유의성 주장은 하지 않는다.')

방법,완료/8,평가 조합 수,상한 계산 수,총 탐색시간(초)
exhaustive,8,4931,0,5.431
branch_bound,8,841,4521,6.5408


가지치기 평가 횟수 감소: 82.94%
각 조건 1회 측정이며 4개 그래프다. 성능 일반화나 유의성 주장은 하지 않는다.


## 해석과 후속 과제
수요·공격을 이용한 방법의 기준선 대비 개선은 확인 표본에서 관찰되었으나 알고리즘 자체도 다르므로 정보만의 인과 효과로 해석하지 않습니다. 중심성 변경, 단일 간선 대비 묶음 복구, 국소 교환의 추가 이득은 제한적입니다. 대리 목적함수 개선이 공식 전달률 개선을 보장하지 않습니다.

가지치기는 평가 횟수를 크게 줄였으나 상한 계산 비용 등 추가 작업이 있어 이번 총 탐색시간은 증가했습니다. 코드 프로파일링 없이 원인을 상한 계산 하나로 확정하지 않습니다.

확인 표본은 합성 그래프 16개이며 완전탐색 완료 가능 조건으로 선정된 문제은행입니다. n14는 개발 표본에만 있습니다. 95% bootstrap 구간은 기술적 불확실성 표시이고 다중검정을 보정한 유의성 검정이 아닙니다. 실행시간은 결정적 프로그램별 1회로, 프로세스 시작 비용과 시스템 변동이 포함됩니다.

실제 조원은 역할별 문헌 원문을 확인하고 코드를 설명·수정·직접 실행한 내역을 개인 기여로 작성해야 합니다. AI가 작성한 부분을 사람의 직접 작성으로 표시하지 않습니다.